# Clasificación de sentimiento — Parte 1
## Experimento: regularización del componente estructural

### Objetivo

Comparar el componente estructural original del modelo v3b con una variante
de menor complejidad, manteniendo las mismas reglas de extracción de
características y las mismas particiones de validación.

### Hipótesis

Reducir la complejidad de los árboles y aumentar la regularización podría
disminuir el ajuste a patrones poco frecuentes y mejorar la generalización.

### Alcance

Primero evaluaremos el componente por separado. Una mejora individual
no garantiza que mejore el ensamble completo ni el score de Kaggle.

El modelo y el submission actuales del grupo se conservarán sin cambios.

## 1. Configuración y entorno

Registramos las versiones de las librerías y fijamos las semillas para
comparar las configuraciones sobre las mismas particiones de los datos.

El experimento utiliza únicamente `train.csv`; no necesitamos cargar
los datos de evaluación de Kaggle.

In [24]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import sklearn

from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
SEEDS = [42, 2024]
LABELS = ["negativo", "neutral", "positivo"]

ROOT = Path.cwd()
if not (ROOT / "data" / "train.csv").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
SOURCE_NOTEBOOK = (
    ROOT / "notebooks" / "modelo_v3b_estructural_polaridad.ipynb"
)

assert (DATA_DIR / "train.csv").exists(), "No se encuentra train.csv."
assert SOURCE_NOTEBOOK.exists(), "No se encuentra el notebook original v3b."

print("Python:", sys.version.split()[0])
print("scikit-learn:", sklearn.__version__)
print("NumPy:", np.__version__)
print("pandas:", pd.__version__)
print("\nProyecto:", ROOT)
print("Notebook original:", SOURCE_NOTEBOOK.name)

Python: 3.12.4
scikit-learn: 1.4.2
NumPy: 1.26.4
pandas: 2.2.2

Proyecto: /Users/alejandrobenavidesrubio/MLP1
Notebook original: modelo_v3b_estructural_polaridad.ipynb


## 2. Carga y revisión de datos

Utilizamos las reseñas etiquetadas de `train.csv`. Comprobamos las columnas,
los valores nulos y la distribución de clases antes de preparar la
validación cruzada.

Los identificadores se conservarán para analizar errores, pero no se
utilizarán como características del modelo.

In [25]:
train_df = pd.read_csv(DATA_DIR / "train.csv")

assert {"id", "text", "label"}.issubset(train_df.columns)
assert train_df[["id", "text", "label"]].notna().all().all()
assert train_df["id"].is_unique
assert set(train_df["label"].unique()) == set(LABELS)
assert train_df["text"].str.strip().ne("").all()

X = train_df["text"].to_numpy()
y = train_df["label"].to_numpy()

print("Dimensiones:", train_df.shape)
print("\nReseñas por clase:")
print(train_df["label"].value_counts())

print("\nTextos duplicados:", train_df["text"].duplicated().sum())

train_df.head(3)

Dimensiones: (12000, 3)

Reseñas por clase:
label
positivo    4212
negativo    4212
neutral     3576
Name: count, dtype: int64

Textos duplicados: 0


,id,text,label
0,0,Lo pedí un martes por la noche y me llegó al d...,neutral
1,1,Se lo compré a un familiar después de pensarla...,positivo
2,2,Lo pedí a mediados de mes y me llegó en tres d...,negativo


## 3. Normalización del texto

Utilizamos la misma función de normalización del modelo v3b: convertimos
a minúsculas y eliminamos las marcas diacríticas mediante Unicode NFKD.

Esto unifica variantes como “batería” y “bateria”. También convierte “ñ”
en “n”. Conservamos la puntuación, los conectores y las negaciones,
porque se utilizan posteriormente para identificar la estructura de la reseña.

Esta transformación aplica reglas fijas y no aprende de las etiquetas.

In [26]:
import unicodedata

def normalizar(texto):
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    return "".join(
        caracter
        for caracter in texto
        if not unicodedata.combining(caracter)
    )


ejemplo = "La batería es EXCELENTE, pero la cámara resultó decepcionante."

print("Original:")
print(ejemplo)

print("\nNormalizado:")
print(normalizar(ejemplo))

Original:
La batería es EXCELENTE, pero la cámara resultó decepcionante.

Normalizado:
la bateria es excelente, pero la camara resulto decepcionante.


### 3.1 Corrección de errores de escritura

Reutilizamos el corrector del modelo v3b. Las palabras frecuentes se
conservan; para las palabras poco frecuentes de más de tres letras,
se buscan alternativas conocidas a una edición de distancia.

Si existen varias candidatas, se elige la más frecuente en entrenamiento.
Los empates se resuelven en orden alfabético.

El vocabulario y las frecuencias se aprenderán únicamente con los datos
de entrenamiento de cada fold. No se ajustará el corrector con todo el
dataset antes de la validación cruzada.

In [27]:
import re
import collections

LETRAS = "abcdefghijklmnopqrstuvwxyzñ"


def _ediciones1(w):
    cortes = [(w[:i], w[i:]) for i in range(len(w) + 1)]

    borrados = [
        a + b[1:]
        for a, b in cortes if b
    ]
    transpuestos = [
        a + b[1] + b[0] + b[2:]
        for a, b in cortes if len(b) > 1
    ]
    sustituidos = [
        a + c + b[1:]
        for a, b in cortes if b
        for c in LETRAS
    ]
    insertados = [
        a + c + b
        for a, b in cortes
        for c in LETRAS
    ]

    return set(borrados + transpuestos + sustituidos + insertados)


class CorrectorTypos:
    def __init__(self, min_frec=5):
        self.min_frec = min_frec

    def fit(self, textos_normalizados):
        self.frecuencia = collections.Counter(
            palabra
            for texto in textos_normalizados
            for palabra in re.findall(r"[a-zñ]+", texto)
        )

        self.vocabulario = {
            palabra
            for palabra, cantidad in self.frecuencia.items()
            if cantidad >= self.min_frec
        }
        self.cache = {}
        return self

    def palabra(self, w):
        if w in self.vocabulario or len(w) <= 3:
            return w

        if w not in self.cache:
            candidatas = sorted(
                candidata
                for candidata in _ediciones1(w)
                if candidata in self.vocabulario
            )

            self.cache[w] = (
                max(candidatas, key=lambda v: self.frecuencia[v])
                if candidatas else w
            )

        return self.cache[w]

    def transform(self, texto):
        return re.sub(
            r"[a-zñ]+",
            lambda m: self.palabra(m.group()),
            texto,
        )

# Expresiones que preceden a un adjetivo de opinión.
COPULAS = [
    "me parecio", "me resulto ser", "me resulto", "resulto ser",
    "resulto", "quedo", "se ve", "se siente", "se sintio",
    "se mantuvo", "termino siendo", "luce", "es", "esta", "fue",
    "me salio", "salio", "me quedo", "anda", "se nota", "se volvio",
    "funciona", "sigue siendo", "sigue", "me parece", "parece",
    "me ha parecido", "se ha mantenido", "me termino pareciendo",
    "termino pareciendo", "vino", "me resulta", "resulta",
    "queda", "se sigue viendo", "estuvo",
]

# [oa] permite reconocer las terminaciones masculina y femenina.
ADJETIVOS_POS = [
    "robust[oa]", "correct[oa]", "nitid[oa]", "impecable",
    "liger[oa]", "sobresaliente", "optim[oa]", "redond[oa]",
    "durable", "resistente", "excelente", "espectacular",
    "comodisim[oa]", "superior", "decente", "potente", "agradable",
    "estupend[oa]", "eficiente", "fiable", "practic[oa]",
    "solid[oa]", "buenisim[oa]", "precis[oa]", "confiable",
    "cumplidor[a]?", "formidable", "comod[oa]", "rapidisim[oa]",
    "veloz", "funcional", "complet[oa]", "buen[oa]", "notable",
    "bien hech[oa]", "bien lograd[oa]", "bien construid[oa]",
    "bien terminad[oa]", "lograd[oa]", "de maravilla", "genial",
    "perfect[oa]", "ideal", "brillante", "increible",
    "fantastic[oa]", "bien", "de buena calidad", "de calidad",
    "muy buen[oa]",
]

ADJETIVOS_NEG = [
    "defectuos[oa]", "imprecis[oa]", "ordinari[oa]", "incomod[oa]",
    "chapucer[oa]", "aparatos[oa]", "terrible", "inconsistente",
    "malisim[oa]", "floj[oa]", "pobre", "olvidable", "debil",
    "lentisim[oa]", "limitad[oa]", "inestable", "torpe",
    "deficiente", "delicad[oa]", "mediocre", "basic[oa]",
    "pesim[oa]", "fragil", "ruidos[oa]", "incumplidor[a]?",
    "desagradable", "corriente", "endeble", "decepcionante",
    "mal hech[oa]", "mal terminad[oa]", "mal construid[oa]",
    "horrible", "bastante mal", "mal", "lent[oa]", "malo",
    "mala", "de mala calidad", "de lo peor",
]

# Modificador opcional entre el verbo y el adjetivo.
MODIFICADORES = (
    r"(?:(?P<mod>muy|bastante|demasiado|super|algo|un poco|"
    r"poco|nada|tan|re)\s+)?"
)

# Expresiones que preceden a un adjetivo de opinión.
COPULAS = [
    "me parecio", "me resulto ser", "me resulto", "resulto ser",
    "resulto", "quedo", "se ve", "se siente", "se sintio",
    "se mantuvo", "termino siendo", "luce", "es", "esta", "fue",
    "me salio", "salio", "me quedo", "anda", "se nota", "se volvio",
    "funciona", "sigue siendo", "sigue", "me parece", "parece",
    "me ha parecido", "se ha mantenido", "me termino pareciendo",
    "termino pareciendo", "vino", "me resulta", "resulta",
    "queda", "se sigue viendo", "estuvo",
]

# [oa] permite reconocer las terminaciones masculina y femenina.
ADJETIVOS_POS = [
    "robust[oa]", "correct[oa]", "nitid[oa]", "impecable",
    "liger[oa]", "sobresaliente", "optim[oa]", "redond[oa]",
    "durable", "resistente", "excelente", "espectacular",
    "comodisim[oa]", "superior", "decente", "potente", "agradable",
    "estupend[oa]", "eficiente", "fiable", "practic[oa]",
    "solid[oa]", "buenisim[oa]", "precis[oa]", "confiable",
    "cumplidor[a]?", "formidable", "comod[oa]", "rapidisim[oa]",
    "veloz", "funcional", "complet[oa]", "buen[oa]", "notable",
    "bien hech[oa]", "bien lograd[oa]", "bien construid[oa]",
    "bien terminad[oa]", "lograd[oa]", "de maravilla", "genial",
    "perfect[oa]", "ideal", "brillante", "increible",
    "fantastic[oa]", "bien", "de buena calidad", "de calidad",
    "muy buen[oa]",
]

ADJETIVOS_NEG = [
    "defectuos[oa]", "imprecis[oa]", "ordinari[oa]", "incomod[oa]",
    "chapucer[oa]", "aparatos[oa]", "terrible", "inconsistente",
    "malisim[oa]", "floj[oa]", "pobre", "olvidable", "debil",
    "lentisim[oa]", "limitad[oa]", "inestable", "torpe",
    "deficiente", "delicad[oa]", "mediocre", "basic[oa]",
    "pesim[oa]", "fragil", "ruidos[oa]", "incumplidor[a]?",
    "desagradable", "corriente", "endeble", "decepcionante",
    "mal hech[oa]", "mal terminad[oa]", "mal construid[oa]",
    "horrible", "bastante mal", "mal", "lent[oa]", "malo",
    "mala", "de mala calidad", "de lo peor",
]

# Modificador opcional entre el verbo y el adjetivo.
MODIFICADORES = (
    r"(?:(?P<mod>muy|bastante|demasiado|super|algo|un poco|"
    r"poco|nada|tan|re)\s+)?"
)

# Detecta y separa: expresión de enlace + modificador + adjetivo.
RE_ADJETIVO = re.compile(
    r"\b(?P<cop>"
    + "|".join(sorted(COPULAS, key=len, reverse=True))
    + r")\s+"
    + MODIFICADORES
    + r"(?P<adj>"
    + "|".join(
        sorted(ADJETIVOS_POS + ADJETIVOS_NEG, key=len, reverse=True)
    )
    + r")\b"
)

# Comprueba si el adjetivo pertenece al inventario positivo.
RE_ADJETIVO_POSITIVO = re.compile(
    r"^(?:" + "|".join(ADJETIVOS_POS) + r")$"
)

print("Inventario de adjetivos y reglas preparados.")
# Demostración independiente: no utiliza los datos del proyecto.
corrector_demo = CorrectorTypos(min_frec=2).fit([
    "el producto es excelente",
    "el producto es excelente",
])

texto_demo = "el producto es exce lente"
print("Palabra original:", "excelennte")
print("Palabra corregida:", corrector_demo.transform("excelennte"))

Inventario de adjetivos y reglas preparados.
Palabra original: excelennte
Palabra corregida: excelente


## 4. Identificación de opiniones
### 4.1 Opiniones expresadas mediante adjetivos

Reutilizamos el inventario del modelo v3b para detectar expresiones como
“la batería resultó excelente” o “la cámara se ve defectuosa”.

La regla identifica un verbo o expresión de enlace, un modificador opcional
y un adjetivo. Las listas asignan una polaridad inicial que posteriormente
podrá complementarse con información aprendida del entrenamiento.

Este inventario se mantiene fijo durante el experimento. Detectar una
expresión no equivale todavía a clasificar el sentimiento de toda la reseña.

In [28]:
ejemplos = [
    "La batería resultó excelente.",
    "La cámara se ve defectuosa.",
    "La carcasa me pareció poco resistente.",
]

for texto in ejemplos:
    coincidencia = RE_ADJETIVO.search(normalizar(texto))

    print(f"\nTexto: {texto}")
    if coincidencia:
        print("Expresión detectada:", coincidencia.group())
        print("Partes:", coincidencia.groupdict())
    else:
        print("No se detectó una expresión del inventario.")


Texto: La batería resultó excelente.
Expresión detectada: resulto excelente
Partes: {'cop': 'resulto', 'mod': None, 'adj': 'excelente'}

Texto: La cámara se ve defectuosa.
Expresión detectada: se ve defectuosa
Partes: {'cop': 'se ve', 'mod': None, 'adj': 'defectuosa'}

Texto: La carcasa me pareció poco resistente.
Expresión detectada: me parecio poco resistente
Partes: {'cop': 'me parecio', 'mod': 'poco', 'adj': 'resistente'}


### 4.2 Opiniones expresadas mediante frases hechas

Conservamos el inventario de expresiones del modelo v3b. Cada expresión
tiene una clave que comienza con `p_` o `n_`, según su polaridad semántica
inicial.

Las expresiones regulares permiten reconocer distintas formas de una misma
frase. Esta polaridad inicial no determina por sí sola la etiqueta de
la reseña completa: una reseña puede contener opiniones contrapuestas.

In [29]:
FRASES_FIJAS = {
    "p_cumple_sobra": r"cumple de sobra",
    "p_vida_facil": r"me hizo la vida (?:mas|muy) facil",
    "p_sorprendio_bien": r"me sorprendio para bien",
    "p_cada_peso": (
        r"cada peso que pague por .{0,40}? valio la pena"
        r"|cada peso que pague valio la pena|valio cada peso"
    ),
    "p_me_encanto": r"\bme encanto\b|\bme encantaron\b",
    "p_feliz": (
        r"\bestoy feliz con|\btermine feliz con|\bquede feliz con"
        r"|me tiene feliz|me dejo feliz|estoy feliz"
        r"|sigo estando feliz|feliz con"
    ),
    "p_contento": (
        r"me tiene content[oa]|me dejo content[oa]|muy content[oa]"
        r"|quede content[oa]|estoy content[oa]"
    ),
    "p_satisfecho": (
        r"mas que satisfech[oa]|quede satisfech[oa]|muy satisfech[oa]"
    ),
    "p_repetiria": r"repetiria la compra",
    "p_recomiendo_sin": (
        r"l[oa] recomiendo sin (?:pensarlo|dudarlo)"
        r"|recomiendo .{0,30}?sin dudarlo"
    ),
    "p_mucho_mejor": r"mucho mejor de lo que",
    "p_me_alegra": (
        r"me alegra haber elegido|me alegra haberl[oa] elegido"
        r"|me alegra haberl[oa] comprado"
    ),
    "p_tal_como": r"respondio tal como esperaba|tal como esperaba",
    "p_sola_queja": r"ni una sola queja",
    "p_supero": r"supero todas mis expectativas|supero mis expectativas",
    "p_vale_pena": r"vale totalmente la pena|vale la pena",
    "p_cinco_estrellas": r"cinco estrellas",
    "p_ojos_cerrados": r"con los ojos cerrados",
    "p_encantado": (
        r"quede encantad[oa] con|quede encantad[oa]"
        r"|quedo encantad[oa]|me quedo encantad[oa]"
    ),
    "p_sin_pensarlo": (
        r"volveria a comprar\w* .{0,40}?sin pensarlo"
        r"|volveria a comprar\w* sin pensarlo|sin pensarlo dos veces"
    ),
    "p_justo_buscando": r"justo lo que (?:estaba buscando|buscaba|necesitaba)",
    "p_acierto": r"todo un acierto",
    "p_encima_media": r"por encima de la media",
    "p_maravilla": r"funciona de maravilla|de maravilla",
    "p_dejo_encantado": r"me dejo encantad[oa]|me tiene encantad[oa]",
    "p_volvi_comprar": r"volvi a comprar",
    "p_con_creces": r"cumple con creces|con creces",
    "p_ni_una_queja": r"ni una queja|una sola queja|no tengo queja|no me quejo",
    "p_me_gusto": (
        r"me gusto (?:bastante|mucho)|me gustaron|me termino gustando"
        r"|termino gustandome|si me gusto|me gusto"
    ),

    "n_no_vale": r"no vale lo que cuesta",
    "n_dolor_cabeza": r"mas de un dolor de cabeza|dolor de cabeza",
    "n_para_nada": r"no es para nada lo que esperaba|para nada lo que esperaba",
    "n_se_dano": r"se dano antes de lo que esperaba|se dano",
    "n_defraudo": r"defraudo todas mis expectativas|defraudo",
    "n_tire_dinero": r"tire el dinero",
    "n_decepciono": r"me decepciono|me decepcionaron",
    "n_me_arrepiento": (
        r"me arrepiento de haber|me arrepiento es de haber"
        r"|me arrepenti de haber\w*"
    ),
    "n_arrepentido": r"arrepentid[oa] de haber",
    "n_problemas": (
        r"no pare de tener problemas|no para de darme problemas"
        r"|no paro de darme problemas|me ha dado problemas"
    ),
    "n_no_convencio": r"no me convencio|no me convence|no me termino de convencer",
    "n_mal_rato": r"mas de un mal rato",
    "n_debajo_aceptable": r"por debajo de lo aceptable",
    "n_peor_parece": r"las cosas que peor",
    "n_molesto": r"quede muy molest[oa]|quede molest[oa]",
    "n_no_recomiendo": (
        r"no l[oa] le recomiendo|no le recomiendo|no se l[oa]s? recomiendo"
        r"|no l[oa] recomiendo|no se l[oa]s? recomendaria|recomendaria a nadie"
    ),
    "n_dejo_molesto": r"me dejo (?:muy )?molest[oa]",
    "n_complicaciones": (
        r"me trajo complicaciones|me dio problemas|dio problemas|me trajo problemas"
    ),
    "n_no_funciono": r"no me funciono como esperaba|nunca me funciono como esperaba",
    "n_desilusiono": r"me desilusiono",
    "n_fue_error": r"fue un error",
    "n_una_estrella": r"una sola estrella|una estrella",
    "n_queda_corto": r"se queda muy cort[oa]|se queda cort[oa]",
    "n_ni_regalado": (
        r"ni regalad[oa]|ni aunque me l[oa] regal\w*|ni me l[oa] regalaron"
    ),
    "n_harto": r"termine hart[oa]|quede hart[oa]|termino hart[oa]",
    "n_error_compra": r"un error de compra",
    "n_debajo_media": r"por debajo de la media",
    "n_bastante_mal": r"funciona bastante mal",
    "n_no_gusto": r"no me gusto nada|no me gusto",
    "n_de_lo_peor": r"de lo peor",
}

RE_FRASES_FIJAS = [
    (clave, re.compile(r"\b(?:" + patron + r")"))
    for clave, patron in FRASES_FIJAS.items()
]

# Demostración: una reseña puede contener señales de ambos signos.
ejemplo = normalizar(
    "La batería superó mis expectativas, pero la cámara no vale lo que cuesta."
)

for clave, patron in RE_FRASES_FIJAS:
    coincidencia = patron.search(ejemplo)
    if coincidencia:
        print(f"{clave}: {coincidencia.group()}")

p_supero: supero mis expectativas
n_no_vale: no vale lo que cuesta


### 4.3 Conectores y cierres ambiguos

Conservamos las listas del modelo v3b para reconocer conectores y frases
de cierre como “ni fu ni fa” o “tengo sentimientos encontrados”.

Estas señales ayudan a describir cómo se relacionan las

In [30]:
CIERRES_AMBIVALENTES = {
    "c_juzgue": (
        r"cada quien que juzgue|que cada quien juzgue|cada quien juzgue"
    ),
    "c_gustos": r"para gustos,? colores",
    "c_dudas": (
        r"sigo con (?:dudas|la duda)|con la duda|me quede con la duda"
        r"|sigo dudando|dudando"
    ),
    "c_fufa": r"ni fu ni fa",
    "c_criterio_cada": r"a criterio de cada uno",
    "c_criterio_tu": r"a tu criterio|a su criterio|a criterio",
    "c_no_sabria": (
        r"no sabria (?:bien )?que (?:recomendar|pensar|decir)"
        r"|no se bien que (?:pensar|decir)"
        r"|no se que (?:pensar|decir)"
    ),
    "c_hay_todo": r"hay de todo",
    "c_sopesalo": r"sopesalo|sopesenlo",
    "c_no_decido": (
        r"no me decido|no me termino de decidir"
        r"|sin decidirme|no termino de decidir"
    ),
    "c_mejor_peor": r"ni lo mejor ni lo peor",
    "c_bueno_malo": r"ni tan bueno ni tan malo",
    "c_ahi_dejo": r"ahi lo dejo|ahi queda",
    "c_sentimientos": r"sentimientos encontrados",
    "c_conclusiones": (
        r"sacara sus conclusiones|saque sus conclusiones"
        r"|sus propias conclusiones"
    ),
    "c_otras_no": (
        r"otras no|lo uno y lo otro|unas cosas? (?:buenas?|mejor)"
        r"|unos detalles mejor que otros"
    ),
    "c_vueltas": r"dandole vueltas|vuelta al asunto",
}

RE_CIERRES = [
    (clave, re.compile(r"\b(?:" + patron + r")"))
    for clave, patron in CIERRES_AMBIVALENTES.items()
]

CONECTORES_PLANTILLA = {
    "fuerte": [
        "pero al final", "pero", "aun asi", "aun con eso", "con todo",
        "al final de cuentas", "al final", "a fin de cuentas",
        "sin embargo", "no obstante", "de todos modos",
        "de todas formas", "lo que si", "ahora bien", "aunque",
        "el detalle es que", "lo curioso es que", "mientras que",
        "eso no quita",
    ],
    "debil": [
        "eso si", "igual", "total", "en fin", "en cambio", "ahora",
    ],
    "aditivo": [
        "por otro lado", "por otra parte", "ademas", "de paso",
        "por cierto", "tambien", "aparte", "encima", "y",
    ],
}

# Buscar primero expresiones largas: "pero al final" antes que "pero".
_LISTA_CONECTORES = sorted(
    [
        (conector, tipo)
        for tipo, conectores in CONECTORES_PLANTILLA.items()
        for conector in conectores
    ],
    key=lambda elemento: -len(elemento[0]),
)

RE_CONECTOR_PLANTILLA = re.compile(
    r"\b("
    + "|".join(re.escape(conector) for conector, _ in _LISTA_CONECTORES)
    + r")\b"
)

TIPO_CONECTOR_PLANTILLA = {
    conector: tipo
    for conector, tipo in _LISTA_CONECTORES
}

# Demostración.
ejemplo = normalizar(
    "La batería es excelente. "
    "Sin embargo, la cámara es defectuosa. Ni fu ni fa."
)

print("Conectores:")
for coincidencia in RE_CONECTOR_PLANTILLA.finditer(ejemplo):
    conector = coincidencia.group()
    print(f"  {conector}: {TIPO_CONECTOR_PLANTILLA[conector]}")

print("\nCierres:")
for clave, patron in RE_CIERRES:
    coincidencia = patron.search(ejemplo)
    if coincidencia:
        print(f"  {clave}: {coincidencia.group()}")

Conectores:
  sin embargo: fuerte

Cierres:
  c_fufa: ni fu ni fa


## 5. Análisis de la estructura de la reseña
### 5.1 Separación del texto y detección de opiniones

Dividimos las reseñas en oraciones y fragmentos. En cada fragmento buscamos
las frases hechas y las expres

In [31]:
def oraciones_plantilla(t):
    return [
        s for s in re.split(r"(?<=[.!?])\s+", t.strip()) if s
    ] or [""]


RE_FRAGMENTO = re.compile(
    r",\s+|;\s+"
    r"|\s+y\s+(?=(?:el|la|los|las|lo|le|me|no|estoy|quede|"
    r"siento|termine|recomiendo|volveria|cada|ni)\b)"
    r"|\s+(?=(?:pero|aunque|mientras que|sin embargo)\b)"
)


def fragmentos_oracion(s):
    # Cada elemento indica: inicio, fin y separador previo.
    salida, pos, sep = [], 0, ""

    for m in RE_FRAGMENTO.finditer(s):
        salida.append((pos, m.start(), sep))
        sep = m.group(0).strip() or " "
        pos = m.end()

    salida.append((pos, len(s), sep))

    return [
        (a, b, separador)
        for a, b, separador in salida
        if s[a:b].strip(" .!?")
    ]


def opiniones_fragmento(f):
    hallazgos = []

    # Detectar frases hechas.
    for clave, patron in RE_FRASES_FIJAS:
        for m in patron.finditer(f):
            hallazgos.append((clave, m.start(), m.end()))

    # Detectar expresiones de verbo + adjetivo.
    for m in RE_ADJETIVO.finditer(f):
        adj = m.group("adj")

        # Evitar interpretar "está bien empacado" como opinión.
        if adj in ("bien", "mal") and m.group("cop") in (
            "es", "esta", "fue", "sigue"
        ):
            continue

        signo = "a+" if RE_ADJETIVO_POSITIVO.match(adj) else "a-"
        mod = m.group("mod") or ""

        if mod in ("poco", "nada"):
            signo = "a-" if signo == "a+" else "a+"

        elif re.search(
            r"\bno\s+$",
            f[max(0, m.start() - 4):m.start()],
        ):
            signo = "a-" if signo == "a+" else "a+"
            mod = "no"

        base = re.sub(r"[oa]$", "", adj) if " " not in adj else adj
        modificador = mod + " " if mod in ("poco", "nada", "no") else ""

        clave = f"{signo}:{modificador}{base}"
        hallazgos.append((clave, m.start(), m.end()))

    # Ordenar por posición; ante el mismo inicio, preferir la más larga.
    hallazgos.sort(key=lambda h: (h[1], -(h[2] - h[1])))

    salida, fin = [], -1
    for hallazgo in hallazgos:
        if hallazgo[1] >= fin:
            salida.append(hallazgo)
            fin = hallazgo[2]

    return salida

In [32]:
ejemplo = normalizar(
    "La batería resultó excelente, pero la carcasa me pareció poco resistente."
)

for oracion in oraciones_plantilla(ejemplo):
    for inicio, fin, separador in fragmentos_oracion(oracion):
        fragmento = oracion[inicio:fin].strip()
        print(f"\nFragmento: {fragmento}")

        for clave, a, b in opiniones_fragmento(fragmento):
            print(f"  Opinión: {fragmento[a:b]}")
            print(f"  Clave: {clave}")


Fragmento: la bateria resulto excelente
  Opinión: resulto excelente
  Clave: a+:excelente

Fragmento: pero la carcasa me parecio poco resistente.
  Opinión: me parecio poco resistente
  Clave: a-:poco resistente


### 5.2 Relación entre opiniones, conectores y cierre

Recorremos los fragmentos para registrar las opiniones en orden. Para cada
opinión conservamos su oración, expresión detectada y contexto cercano.

Identificamos el último conector del

In [33]:
def analizar_resena(t):
    """Recibe texto normalizado y devuelve su estructura."""
    S = oraciones_plantilla(t)
    opiniones = []

    for si, s in enumerate(S):
        fin_anterior = 0

        for a, b, sep in fragmentos_oracion(s):
            f = s[a:b]
            nucleos = opiniones_fragmento(f)

            if not nucleos:
                continue

            # Si hay varias opiniones, repartir el contexto entre ellas.
            for j, (clave, ca, cb) in enumerate(nucleos):
                nb = nucleos[j + 1][1] if j + 1 < len(nucleos) else len(f)
                pa = nucleos[j - 1][2] if j > 0 else 0

                opiniones.append({
                    "oracion": si,
                    "clave": clave,
                    "nucleo": f[ca:cb],
                    "inicio_fragmento": f[pa:ca],
                    "resto_fragmento": f[cb:nb],
                    "separador": sep if j == 0 else "y",
                    "antes": s[fin_anterior:a] if j == 0 else "",
                    "ini": a + pa,
                    "fin": a + nb,
                    "oracion_texto": s,
                })

            fin_anterior = b

    # Añadir contexto posterior y conector previo.
    for j, opinion in enumerate(opiniones):
        siguiente = (
            opiniones[j + 1]
            if (
                j + 1 < len(opiniones)
                and opiniones[j + 1]["oracion"] == opinion["oracion"]
            )
            else None
        )

        opinion["despues"] = opinion["oracion_texto"][
            opinion["fin"]:
            siguiente["ini"] if siguiente else len(opinion["oracion_texto"])
        ]

        contexto_previo = (
            opinion["antes"] + " "
            + opinion["inicio_fragmento"] + " "
            + opinion["separador"]
        )

        conectores = RE_CONECTOR_PLANTILLA.findall(contexto_previo)
        opinion["conector"] = conectores[-1] if conectores else ""
        opinion["tipo_conector"] = TIPO_CONECTOR_PLANTILLA.get(
            opinion["conector"], "ninguno"
        )

    # Buscar un cierre en las oraciones posteriores a la última opinión.
    ultima_oracion = opiniones[-1]["oracion"] if opiniones else -1
    cierre, cierre_oracion = "", -1

    for si in range(len(S) - 1, ultima_oracion, -1):
        for clave, patron in RE_CIERRES:
            if patron.search(S[si]):
                cierre, cierre_oracion = clave, si
                break
        if cierre:
            break

    # O después de la opinión, dentro de la misma oración.
    cierre_en_linea = ""

    if opiniones and not cierre:
        for clave, patron in RE_CIERRES:
            if patron.search(opiniones[-1]["despues"]):
                cierre_en_linea = clave
                break

    return {
        "oraciones": S,
        "opiniones": opiniones,
        "cierre": cierre,
        "cierre_oracion": cierre_oracion,
        "cierre_en_linea": cierre_en_linea,
    }

In [34]:
ejemplo = normalizar(
    "La batería resultó excelente, pero la carcasa me pareció poco resistente. "
    "Ni fu ni fa."
)

analisis_demo = analizar_resena(ejemplo)

for opinion in analisis_demo["opiniones"]:
    print("Opinión:", opinion["nucleo"])
    print("Clave:", opinion["clave"])
    print("Conector:", opinion["conector"] or "(ninguno)")
    print("Tipo:", opinion["tipo_conector"])
    print()

print("Cierre:", analisis_demo["cierre"] or "(ninguno)")

Opinión: resulto excelente
Clave: a+:excelente
Conector: (ninguno)
Tipo: ninguno

Opinión: me parecio poco resistente
Clave: a-:poco resistente
Conector: pero
Tipo: fuerte

Cierre: c_fufa


### 5.3 Polaridad, tipo de opinión y final de la reseña

Asignamos una polaridad inicial de +1 o −1 a cada opinión detectada y
distinguimos las expresiones adjetivales de las frases hechas.

También identificamos si después de la última opinión hay otras oraciones
o un cierre ambiguo. “Relleno” es el nombre utilizado por el parser para
las oraciones posteriores sin opiniones reconocidas; no demuestra que
carezcan de información.

In [35]:
def polaridad_semantica(clave):
    return 1 if (
        clave.startswith("p_") or clave.startswith("a+")
    ) else -1


def es_adjetival(opinion):
    return opinion["clave"][:2] in ("a+", "a-")


def tipo_de_final(analisis):
    # Se utiliza únicamente cuando hay al menos una opinión detectada.
    S = analisis["oraciones"]
    ultima = analisis["opiniones"][-1]

    relleno = (
        len(S) - 1 - ultima["oracion"]
        - (
            1
            if analisis["cierre_oracion"] > ultima["oracion"]
            else 0
        )
    )

    if analisis["cierre"] or analisis["cierre_en_linea"]:
        return "cierre", relleno

    return ("relleno" if relleno > 0 else "opinion"), relleno


# Aplicar las funciones al ejemplo del paso anterior.
for opinion in analisis_demo["opiniones"]:
    print(
        f"{opinion['nucleo']} → "
        f"polaridad: {polaridad_semantica(opinion['clave']):+d}, "
        f"adjetival: {es_adjetival(opinion)}"
    )

final, n_relleno = tipo_de_final(analisis_demo)
print("\nTipo de final:", final)
print("Oraciones posteriores sin contar la del cierre:", n_relleno)

resulto excelente → polaridad: +1, adjetival: True
me parecio poco resistente → polaridad: -1, adjetival: True

Tipo de final: cierre
Oraciones posteriores sin contar la del cierre: 0


## 6. Construcción de características estructurales

Transformamos el análisis de cada reseña en 59 características numéricas:
conteos de opiniones, polaridades, tipos de conectores y posición de la
última opinión.

Algunas características combinan señales. Por ejemplo, el signo de la
última opinión junto con la presencia de un conector permite distinguir
un contraste que termina en positivo de uno que termina en negativo.

Conservamos las características del modelo v3b para evaluar únicamente
el cambio de regularización del clasificador.

In [36]:
CONECTORES_RASGO = [
    "eso si", "aun asi", "al final", "con todo", "pero",
    "a fin de cuentas", "por otro lado", "de paso", "por cierto",
    "ademas", "ahora", "y", "", "lo que si", "aunque",
    "sin embargo", "pero al final", "igual", "en cambio",
    "ahora bien", "aun con eso",
]


def rasgos_plantilla(analisis, signo=polaridad_semantica):
    ops = analisis["opiniones"]
    S = analisis["oraciones"]
    f = {}

    n = len(ops)
    signos = [signo(o["clave"]) for o in ops]

    # 1. Cantidad y tipo de opiniones.
    f["sin_opinion"] = float(n == 0)
    f["n_opiniones"] = n
    f["n_positivas"] = sum(s == 1 for s in signos)
    f["n_negativas"] = sum(s == -1 for s in signos)
    f["n_adjetivales"] = sum(es_adjetival(o) for o in ops)
    f["n_fijas"] = n - f["n_adjetivales"]
    f["signo_mayoria"] = float(
        np.sign(f["n_positivas"] - f["n_negativas"])
    )

    # 2. Orden y coherencia de las polaridades.
    s_primera = signos[0] if n else 0
    s_ultima = signos[-1] if n else 0

    f["signo_primera"] = s_primera
    f["signo_ultima"] = s_ultima

    coherente = n > 0 and len(set(signos)) == 1
    mixta = len(set(signos)) == 2

    f["coherente"] = float(coherente)
    f["signo_coherente"] = s_ultima if coherente else 0
    f["mixta"] = float(mixta)

    adj = [
        s for s, o in zip(signos, ops)
        if es_adjetival(o)
    ]
    fij = [
        s for s, o in zip(signos, ops)
        if not es_adjetival(o)
    ]

    f["signo_ultima_adj"] = adj[-1] if adj else 0
    f["signo_primera_adj"] = adj[0] if adj else 0
    f["signo_ultima_fija"] = fij[-1] if fij else 0

    f["signo_mixta_A"] = s_ultima if (mixta and adj) else 0
    f["signo_mixta_B"] = s_ultima if (mixta and not adj) else 0

    # 3. Conector del último cambio de signo.
    # Si no hay cambio, se utiliza el de la última opinión.
    jc = (
        [
            j for j in range(1, n)
            if signos[j] != signos[j - 1]
        ][-1]
        if mixta else n - 1
    )

    con = ops[jc]["conector"] if n else ""
    tipo = ops[jc]["tipo_conector"] if n else "ninguno"

    for t in ["fuerte", "debil", "aditivo", "ninguno"]:
        f["conector_" + t] = float(tipo == t)
        f["signo_mixta_conector_" + t] = (
            s_ultima if (tipo == t and mixta) else 0
        )

    for c in CONECTORES_RASGO:
        f["signo_mixta_con=" + (c or "nada")] = (
            s_ultima if (con == c and mixta) else 0
        )

    # 4. Cómo termina la reseña.
    if n:
        ultima = ops[-1]
        final, relleno = tipo_de_final(analisis)

        f["relleno_despues"] = relleno
        f["cierre"] = float(bool(analisis["cierre"]))
        f["cierre_en_linea"] = float(
            bool(analisis["cierre_en_linea"])
        )

        f["termina_opinion"] = float(final == "opinion")
        f["termina_relleno"] = float(final == "relleno")
        f["termina_cierre"] = float(final == "cierre")

        f["signo_ultima_x_termina_opinion"] = (
            s_ultima * f["termina_opinion"]
        )
        f["signo_ultima_x_termina_relleno"] = (
            s_ultima * f["termina_relleno"]
        )
        f["signo_ultima_x_termina_cierre"] = (
            s_ultima * f["termina_cierre"]
        )

        f["posicion_ultima"] = (
            ultima["oracion"] / max(1, len(S) - 1)
        )

        cola = ultima["resto_fragmento"] + " " + ultima["despues"]

        f["signo_ultima_eso_si_cola"] = (
            s_ultima * float(bool(re.search(r"\beso si\b", cola)))
        )
        f["signo_ultima_algun_fuerte"] = (
            s_ultima
            * float(any(o["tipo_conector"] == "fuerte" for o in ops[1:]))
        )

    else:
        for k in [
            "relleno_despues", "cierre", "cierre_en_linea",
            "termina_opinion", "termina_relleno", "termina_cierre",
            "signo_ultima_x_termina_opinion",
            "signo_ultima_x_termina_relleno",
            "signo_ultima_x_termina_cierre",
            "posicion_ultima", "signo_ultima_eso_si_cola",
            "signo_ultima_algun_fuerte",
        ]:
            f[k] = 0.0

    f["n_oraciones"] = len(S)
    return f

In [37]:
rasgos_demo = rasgos_plantilla(analisis_demo)

print("Número de características:", len(rasgos_demo))
assert len(rasgos_demo) == 59

seleccion = [
    "n_opiniones",
    "n_positivas",
    "n_negativas",
    "mixta",
    "signo_ultima",
    "conector_fuerte",
    "termina_cierre",
]

pd.Series(rasgos_demo).loc[seleccion].to_frame("valor")

Número de características: 59


,valor
n_opiniones,2.0
n_positivas,1.0
n_negativas,1.0
mixta,1.0
signo_ultima,-1.0
conector_fuerte,1.0
termina_cierre,1.0


### 6.1 Transformador de características

El transformador reúne la normalización, la corrección de errores de
escritura, el análisis de opiniones y la generación de características.

Durante el entrenamiento aprende la polaridad de expresiones que aparecen
como única opinión en reseñas positivas o negativas. Si una expresión
tiene menos de tres ejemplos, conserva la polaridad semántica del inventario.

En cada fold se ajustará un transformador nuevo utilizando exclusivamente
sus datos de entrenamiento. La validación se procesará con ese mismo
transformador, sin volver a ajustarlo.

In [38]:
from sklearn.base import BaseEstimator, TransformerMixin


def _es_positivo(v):
    # Admite etiquetas de texto o codificadas: 2 = positivo.
    return v == "positivo" or (
        not isinstance(v, str) and int(v) == 2
    )


def _es_negativo(v):
    # Admite etiquetas de texto o codificadas: 0 = negativo.
    return v == "negativo" or (
        not isinstance(v, str) and int(v) == 0
    )


class RasgosPlantilla(TransformerMixin, BaseEstimator):
    def __init__(
        self,
        min_frec=5,
        modo_polaridad="aprendida",
        min_n=3,
    ):
        self.min_frec = min_frec
        self.modo_polaridad = modo_polaridad
        self.min_n = min_n

    def _ajustar(self, X, y):
        self.corrector_ = CorrectorTypos(self.min_frec).fit(
            [normalizar(t) for t in X]
        )

        self.lexico_ = {}
        analisis = self.analizar(X)

        if self.modo_polaridad == "aprendida":
            conteo = {}

            for a, etiqueta in zip(analisis, y):
                if (
                    len(a["opiniones"]) != 1
                    or not (
                        _es_positivo(etiqueta)
                        or _es_negativo(etiqueta)
                    )
                ):
                    continue

                clave = a["opiniones"][0]["clave"]
                conteo.setdefault(clave, [0, 0])
                conteo[clave][int(_es_positivo(etiqueta))] += 1

            self.lexico_ = {
                clave: (1 if positivos > negativos else -1)
                for clave, (negativos, positivos) in conteo.items()
                if negativos + positivos >= self.min_n
            }

        return analisis

    def fit(self, X, y=None):
        self._ajustar(X, y)
        return self

    def fit_transform(self, X, y=None):
        # Analiza entrenamiento una sola vez.
        return self._tabla(self._ajustar(X, y))

    def signo(self, clave):
        return self.lexico_.get(
            clave,
            polaridad_semantica(clave),
        )

    def analizar(self, X):
        return [
            analizar_resena(
                self.corrector_.transform(normalizar(t))
            )
            for t in X
        ]

    def _tabla(self, analisis):
        return pd.DataFrame([
            rasgos_plantilla(a, self.signo)
            for a in analisis
        ]).astype(float)

    def transform(self, X):
        return self._tabla(self.analizar(X))


print("Transformador preparado.")

Transformador preparado.


## 7. Configuraciones del clasificador

Comparamos el componente estructural original con dos variantes de menor
complejidad. Las características y las particiones de validación serán
iguales para los tres modelos.

| Configuración | Máximo de hojas por árbol | Mínimo de ejemplos por hoja | Regularización L2 |
|---|---:|---:|---:|
| Original | 15 | 20 | 0 |
| Regularizado | 7 | 60 | 5 |
| Conservador | 3 | 100 | 10 |

Conservamos la tasa de aprendizaje de 0,05 y las 200 iteraciones del
modelo original. Las configuraciones se fijan antes de evaluar sus resultados.

In [39]:
from threadpoolctl import threadpool_limits

# Mismo límite usado en la prueba de referencia de este experimento.
threadpool_limits(limits=2)

CONFIGURACIONES = {
    "original": {
        "max_leaf_nodes": 15,
        "min_samples_leaf": 20,
        "l2_regularization": 0,
    },
    "regularizado": {
        "max_leaf_nodes": 7,
        "min_samples_leaf": 60,
        "l2_regularization": 5,
    },
    "conservador": {
        "max_leaf_nodes": 3,
        "min_samples_leaf": 100,
        "l2_regularization": 10,
    },
}


def crear_clasificador(nombre):
    return HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=200,
        early_stopping=False,
        random_state=0,
        **CONFIGURACIONES[nombre],
    )


# Preparar las mismas particiones para las tres configuraciones.
PARTICIONES = {
    semilla: list(
        StratifiedKFold(
            n_splits=5,
            shuffle=True,
            random_state=semilla,
        ).split(X, y)
    )
    for semilla in SEEDS
}

display(pd.DataFrame(CONFIGURACIONES).T)

entrenamiento, validacion = PARTICIONES[42][0]

print("\nPrimer fold:")
print("Entrenamiento:", len(entrenamiento))
print("Validación:", len(validacion))
assert set(entrenamiento).isdisjoint(validacion)

,max_leaf_nodes,min_samples_leaf,l2_regularization
original,15,20,0
regularizado,7,60,5
conservador,3,100,10



Primer fold:
Entrenamiento: 9600
Validación: 2400


## 8. Validación cruzada

Evaluamos las tres configuraciones con cinco folds estratificados y dos
semillas. En cada fold ajustamos un extractor nuevo con entrenamiento
y lo aplicamos a validación sin consultar sus etiquetas.

Guardamos las predicciones fuera de muestra para comparar después qué
errores corrige o introduce cada configuración. No utilizamos `eval.csv`.

El inventario de reglas procede de un análisis previo de train, por lo que
la evaluación conserva ese sesgo de diseño y debe interpretarse como exploratoria.

In [40]:
import time

resultados_cv = []
predicciones_oof = {}

inicio_total = time.perf_counter()

for semilla, folds in PARTICIONES.items():
    predicciones = {
        nombre: np.empty(len(y), dtype=object)
        for nombre in CONFIGURACIONES
    }
    cobertura = np.zeros(len(y), dtype=int)

    print(f"\n=== Semilla {semilla} ===", flush=True)

    for numero_fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        inicio_fold = time.perf_counter()

        # Aprender corrector y polaridad solo con entrenamiento.
        extractor = RasgosPlantilla(modo_polaridad="aprendida")

        X_tr_rasgos = extractor.fit_transform(X[tr_idx], y[tr_idx])
        X_va_rasgos = extractor.transform(X[va_idx])

        assert X_tr_rasgos.shape[1] == 59
        assert X_tr_rasgos.columns.equals(X_va_rasgos.columns)
        assert np.isfinite(X_tr_rasgos.to_numpy()).all()
        assert np.isfinite(X_va_rasgos.to_numpy()).all()

        resumen_fold = []

        for nombre in CONFIGURACIONES:
            modelo = crear_clasificador(nombre)
            modelo.fit(X_tr_rasgos, y[tr_idx])

            pred = modelo.predict(X_va_rasgos)
            predicciones[nombre][va_idx] = pred

            acc = accuracy_score(y[va_idx], pred)

            resultados_cv.append({
                "semilla": semilla,
                "fold": numero_fold,
                "modelo": nombre,
                "accuracy": acc,
            })

            resumen_fold.append(f"{nombre}: {acc:.4f}")

        cobertura[va_idx] += 1

        segundos = time.perf_counter() - inicio_fold
        print(
            f"Fold {numero_fold}/5 | "
            + " | ".join(resumen_fold)
            + f" | {segundos:.1f} s",
            flush=True,
        )

    # Cada reseña debe haber sido predicha exactamente una vez.
    assert (cobertura == 1).all()
    predicciones_oof[semilla] = predicciones

resultados_cv = pd.DataFrame(resultados_cv)

resumen_cv = (
    resultados_cv
    .groupby(["modelo", "semilla"])["accuracy"]
    .mean()
    .unstack("semilla")
    .reindex(CONFIGURACIONES.keys())
)

print(
    f"\nTiempo total: "
    f"{(time.perf_counter() - inicio_total) / 60:.1f} minutos"
)
print("\nAccuracy promedio por configuración y semilla:")
display(resumen_cv.round(6))


=== Semilla 42 ===
Fold 1/5 | original: 0.9079 | regularizado: 0.9150 | conservador: 0.9087 | 10.2 s
Fold 2/5 | original: 0.9038 | regularizado: 0.9021 | conservador: 0.9046 | 9.1 s
Fold 3/5 | original: 0.9117 | regularizado: 0.9137 | conservador: 0.9142 | 9.2 s
Fold 4/5 | original: 0.9146 | regularizado: 0.9142 | conservador: 0.9154 | 8.8 s
Fold 5/5 | original: 0.8996 | regularizado: 0.9050 | conservador: 0.9117 | 8.0 s

=== Semilla 2024 ===
Fold 1/5 | original: 0.9137 | regularizado: 0.9171 | conservador: 0.9121 | 7.9 s
Fold 2/5 | original: 0.9163 | regularizado: 0.9154 | conservador: 0.9196 | 8.1 s
Fold 3/5 | original: 0.9021 | regularizado: 0.9067 | conservador: 0.9038 | 8.1 s
Fold 4/5 | original: 0.9046 | regularizado: 0.9079 | conservador: 0.9079 | 8.5 s
Fold 5/5 | original: 0.9113 | regularizado: 0.9113 | conservador: 0.9117 | 8.2 s

Tiempo total: 1.4 minutos

Accuracy promedio por configuración y semilla:


semilla,42,2024
modelo,,
original,0.907500,0.909583
regularizado,0.910000,0.911667
conservador,0.910917,0.911000


### 8.1 Comparación de errores frente al componente original

Las dos variantes mejoraron el accuracy promedio en ambas particiones,
aunque no en todos los folds.

Comparamos sus predicciones fuera de muestra con las del componente
original. Contamos los errores corregidos y los aciertos perdidos para
entender qué hay detrás de la mejora neta.

Las dos particiones reutilizan las mismas 12.000 reseñas; no constituyen
dos conjuntos de prueba independientes.

In [41]:
comparaciones = []

for semilla in SEEDS:
    original_correcto = (
        predicciones_oof[semilla]["original"] == y
    )

    for nombre in ["regularizado", "conservador"]:
        nuevo_correcto = predicciones_oof[semilla][nombre] == y

        corregidos = int(
            (~original_correcto & nuevo_correcto).sum()
        )
        introducidos = int(
            (original_correcto & ~nuevo_correcto).sum()
        )

        comparaciones.append({
            "semilla": semilla,
            "modelo": nombre,
            "errores_corregidos": corregidos,
            "errores_introducidos": introducidos,
            "aciertos_netos": corregidos - introducidos,
            "mejora_pp": 100 * (corregidos - introducidos) / len(y),
        })

comparacion_errores = pd.DataFrame(comparaciones)
display(comparacion_errores.round(3))

,semilla,modelo,errores_corregidos,errores_introducidos,aciertos_netos,mejora_pp
0,42,regularizado,186,156,30,0.250
1,42,conservador,357,316,41,0.342
2,2024,regularizado,181,156,25,0.208
3,2024,conservador,311,294,17,0.142


## 9. Entrenamiento final y generación del submission

Seleccionamos la configuración regularizada, que mejoró frente al
componente estructural original con ambas semillas de validación.

Esta entrega evalúa el modelo estructural regularizado por separado.
No corresponde al ensamble completo v3b ni demuestra una mejora de ese
ensamble.

Entrenamos el extractor y el clasificador con todas las reseñas de
train.csv. Luego predecimos eval.csv, respetando los identificadores
y el orden de sample_submission.csv.

Las etiquetas de evaluación no están disponibles y no se utilizan
para entrenar ni seleccionar parámetros.

In [44]:
from sklearn.pipeline import Pipeline

CONFIGURACION_FINAL = "regularizado"

modelo_final = Pipeline([
    (
        "rasgos",
        RasgosPlantilla(modo_polaridad="aprendida"),
    ),
    (
        "clasificador",
        crear_clasificador(CONFIGURACION_FINAL),
    ),
])

inicio = time.perf_counter()

modelo_final.fit(X, y)

print("Modelo final entrenado.")
print("Configuración:", CONFIGURACION_FINAL)
print("Reseñas utilizadas:", len(y))
print("Clases:", modelo_final.classes_)
print(f"Tiempo: {time.perf_counter() - inicio:.1f} segundos")

Modelo final entrenado.
Configuración: regularizado
Reseñas utilizadas: 12000
Clases: ['negativo' 'neutral' 'positivo']
Tiempo: 6.2 segundos


In [45]:
eval_df = pd.read_csv(DATA_DIR / "eval.csv")
sample_submission = pd.read_csv(
    DATA_DIR / "sample_submission.csv"
)

assert {"id", "text"}.issubset(eval_df.columns)
assert list(sample_submission.columns) == ["id", "answer"]

assert eval_df[["id", "text"]].notna().all().all()
assert sample_submission["id"].notna().all()
assert eval_df["text"].str.strip().ne("").all()

assert eval_df["id"].is_unique
assert sample_submission["id"].is_unique
assert len(eval_df) == len(sample_submission)
assert set(eval_df["id"]) == set(sample_submission["id"])

predicciones_eval = modelo_final.predict(
    eval_df["text"].to_numpy()
)

predicciones_por_id = pd.Series(
    predicciones_eval,
    index=eval_df["id"],
)

submission = sample_submission.copy()
submission["answer"] = (
    submission["id"].map(predicciones_por_id)
)

assert submission["answer"].notna().all()
assert set(submission["answer"]).issubset(LABELS)
assert submission["id"].equals(sample_submission["id"])

print("Filas:", len(submission))
print("Columnas:", submission.columns.tolist())
print("\nDistribución de predicciones:")
print(submission["answer"].value_counts())

display(submission.head())

Filas: 3000
Columnas: ['id', 'answer']

Distribución de predicciones:
answer
negativo    1092
positivo    1036
neutral      872
Name: count, dtype: int64


,id,answer
0,0,neutral
1,1,negativo
2,2,neutral
3,3,neutral
4,4,neutral


In [46]:
import joblib
from datetime import datetime

MODELS_DIR = ROOT / "models"
SUBMISSIONS_DIR = ROOT / "submissions"
RESULTS_DIR = ROOT / "results"

for carpeta in (MODELS_DIR, SUBMISSIONS_DIR, RESULTS_DIR):
    carpeta.mkdir(parents=True, exist_ok=True)

experiment_name = "estructural_regularizado"
marca_tiempo = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_ejecucion = f"{experiment_name}_{marca_tiempo}"

ruta_modelo = MODELS_DIR / f"modelo_{nombre_ejecucion}.joblib"
ruta_submission = (
    SUBMISSIONS_DIR / f"submission_{nombre_ejecucion}.csv"
)
ruta_resultados = RESULTS_DIR / f"cv_{nombre_ejecucion}.csv"

for ruta in (ruta_modelo, ruta_submission, ruta_resultados):
    assert not ruta.exists(), f"El archivo ya existe: {ruta}"

joblib.dump(modelo_final, ruta_modelo)
submission.to_csv(ruta_submission, index=False)
resultados_cv.to_csv(ruta_resultados, index=False)

# Recargar los archivos y verificar que conservan las predicciones.
modelo_recargado = joblib.load(ruta_modelo)
submission_guardado = pd.read_csv(ruta_submission)

predicciones_recargadas = pd.Series(
    modelo_recargado.predict(eval_df["text"].to_numpy()),
    index=eval_df["id"],
)

esperadas = submission_guardado["id"].map(
    predicciones_recargadas
)

assert list(submission_guardado.columns) == ["id", "answer"]
assert len(submission_guardado) == len(sample_submission)
assert np.array_equal(
    submission_guardado["id"].to_numpy(),
    sample_submission["id"].to_numpy(),
)
assert np.array_equal(
    submission_guardado["answer"].to_numpy(),
    esperadas.to_numpy(),
)

print("OK: el modelo recargado reproduce el CSV.")
print("\nModelo:", ruta_modelo)
print("Resultados de validación:", ruta_resultados)
print("\nARCHIVO PARA SUBIR A KAGGLE:")
print(ruta_submission)

OK: el modelo recargado reproduce el CSV.

Modelo: /Users/alejandrobenavidesrubio/MLP1/models/modelo_estructural_regularizado_20261005_212407.joblib
Resultados de validación: /Users/alejandrobenavidesrubio/MLP1/results/cv_estructural_regularizado_20261005_212407.csv

ARCHIVO PARA SUBIR A KAGGLE:
/Users/alejandrobenavidesrubio/MLP1/submissions/submission_estructural_regularizado_20261005_212407.csv


## 10. Resultado en Kaggle y conclusión

El modelo estructural regularizado obtuvo un Public Score de 0.87111.
No superó el mejor envío del equipo, correspondiente al modelo v3b.

La mejora del componente en validación cruzada no se tradujo en un
submission superior al ensamble del equipo. Son modelos diferentes:
este experimento evaluó únicamente el componente estructural.

Se conserva v3b como mejor candidato. Queda pendiente evaluar si
reemplazar su componente estructural por la variante regularizada
mejora el ensamble, manteniendo constantes los demás componentes
y sus pesos.

## 11. Experimento: polaridad aprendida frente a polaridad semántica

### Hipótesis

Aprender la polaridad de las expresiones a partir de las etiquetas de
reseñas completas puede introducir asociaciones que no se mantienen
en datos nuevos.

Comparamos la polaridad aprendida con el inventario semántico fijo,
manteniendo el clasificador regularizado y las mismas particiones.

En ambas variantes, el corrector ortográfico se ajusta exclusivamente
con los textos de entrenamiento de cada fold.

Evaluamos cinco folds con las semillas 42 y 2024. Conservamos las
probabilidades fuera de muestra para posteriores comparaciones.
No utilizamos eval.csv para seleccionar la variante.

El inventario procede del trabajo previo sobre train.csv, por lo que
esta evaluación sigue siendo exploratoria.

In [47]:
MODOS_POLARIDAD = ["aprendida", "semantica"]

resultados_polaridad = []
probabilidades_oof_polaridad = {}
clases_polaridad = np.array(LABELS)

inicio_experimento = time.perf_counter()

for semilla, folds in PARTICIONES.items():
    probabilidades_oof_polaridad[semilla] = {
        modo: np.full((len(y), len(LABELS)), np.nan)
        for modo in MODOS_POLARIDAD
    }
    cobertura = np.zeros(len(y), dtype=int)

    print(f"\n=== Semilla {semilla} ===", flush=True)

    for numero_fold, (tr_idx, va_idx) in enumerate(folds, start=1):
        resumen_fold = []

        for modo in MODOS_POLARIDAD:
            candidato = Pipeline([
                (
                    "rasgos",
                    RasgosPlantilla(modo_polaridad=modo),
                ),
                (
                    "clasificador",
                    crear_clasificador("regularizado"),
                ),
            ])

            candidato.fit(X[tr_idx], y[tr_idx])

            # Alinear las columnas con el orden de LABELS.
            orden = [
                list(candidato.classes_).index(clase)
                for clase in clases_polaridad
            ]
            probabilidades = candidato.predict_proba(
                X[va_idx]
            )[:, orden]

            assert np.isfinite(probabilidades).all()
            assert np.allclose(probabilidades.sum(axis=1), 1)

            probabilidades_oof_polaridad[semilla][modo][
                va_idx
            ] = probabilidades

            predicciones = clases_polaridad[
                probabilidades.argmax(axis=1)
            ]
            accuracy = accuracy_score(y[va_idx], predicciones)

            resultados_polaridad.append({
                "semilla": semilla,
                "fold": numero_fold,
                "polaridad": modo,
                "accuracy": accuracy,
            })

            resumen_fold.append(f"{modo}: {accuracy:.4f}")

        cobertura[va_idx] += 1

        print(
            f"Fold {numero_fold}/5 | "
            + " | ".join(resumen_fold),
            flush=True,
        )

    assert (cobertura == 1).all()

    for modo in MODOS_POLARIDAD:
        assert np.isfinite(
            probabilidades_oof_polaridad[semilla][modo]
        ).all()

resultados_polaridad = pd.DataFrame(resultados_polaridad)

resumen_polaridad = (
    resultados_polaridad
    .groupby(["semilla", "polaridad"])["accuracy"]
    .mean()
    .unstack("polaridad")
)

resumen_polaridad["diferencia_pp"] = 100 * (
    resumen_polaridad["semantica"]
    - resumen_polaridad["aprendida"]
)

print("\nResultados:")
display(resumen_polaridad.round(6))

print(
    f"Tiempo total: "
    f"{(time.perf_counter() - inicio_experimento) / 60:.1f} minutos"
)


=== Semilla 42 ===
Fold 1/5 | aprendida: 0.9150 | semantica: 0.9150
Fold 2/5 | aprendida: 0.9021 | semantica: 0.9021
Fold 3/5 | aprendida: 0.9137 | semantica: 0.9137
Fold 4/5 | aprendida: 0.9142 | semantica: 0.9142
Fold 5/5 | aprendida: 0.9050 | semantica: 0.9050

=== Semilla 2024 ===
Fold 1/5 | aprendida: 0.9171 | semantica: 0.9171
Fold 2/5 | aprendida: 0.9154 | semantica: 0.9154
Fold 3/5 | aprendida: 0.9067 | semantica: 0.9067
Fold 4/5 | aprendida: 0.9079 | semantica: 0.9079
Fold 5/5 | aprendida: 0.9113 | semantica: 0.9113

Resultados:


polaridad,aprendida,semantica,diferencia_pp
semilla,,,
42,0.910000,0.910000,0.0
2024,0.911667,0.911667,0.0


Tiempo total: 1.8 minutos


In [48]:
for semilla in SEEDS:
    aprendida = probabilidades_oof_polaridad[semilla]["aprendida"]
    semantica = probabilidades_oof_polaridad[semilla]["semantica"]

    cambios = np.count_nonzero(
        aprendida.argmax(axis=1) != semantica.argmax(axis=1)
    )
    diferencia_maxima = np.abs(aprendida - semantica).max()

    print(f"\nSemilla {semilla}")
    print("Predicciones distintas:", cambios)
    print(
        "Diferencia máxima entre probabilidades:",
        f"{diferencia_maxima:.10f}",
    )


Semilla 42
Predicciones distintas: 0
Diferencia máxima entre probabilidades: 0.0000000000

Semilla 2024
Predicciones distintas: 0
Diferencia máxima entre probabilidades: 0.0000000000


### Conclusión del experimento de polaridad

Las variantes de polaridad aprendida y semántica produjeron las mismas
predicciones y probabilidades fuera de muestra con ambas semillas.

En esta evaluación, cambiar el modo de polaridad no modificó el
comportamiento del modelo. No se genera un nuevo submission.

In [49]:
import ast
import json

with SOURCE_NOTEBOOK.open(encoding="utf-8") as archivo:
    notebook_original = json.load(archivo)

buscadas = {
    "Bloques",
    "modelo_lr_palabras",
    "modelo_stacking",
    "VectorizadorFragmentos",
    "ClasificadorFragmentos",
    "ComponenteEstLR",
    "modelo_est_lr",
}

for numero, celda in enumerate(
    notebook_original["cells"], start=1
):
    if celda["cell_type"] != "code":
        continue

    codigo = "".join(celda["source"])

    try:
        estructura = ast.parse(codigo)
    except SyntaxError:
        continue

    definiciones = [
        elemento.name
        for elemento in estructura.body
        if isinstance(
            elemento,
            (ast.FunctionDef, ast.ClassDef),
        )
    ]

    if buscadas.intersection(definiciones):
        print(f"Celda {numero}:")
        print("  " + ", ".join(definiciones))

Celda 21:
  _regex_conectores, desde_ultimo, Bloques
Celda 24:
  tfidf_palabras, tfidf_caracteres, vectorizar, modelo_lr_palabras, modelo_lr_pal_car, modelo_svc_pal_car, modelo_stacking
Celda 49:
  a_texto, VectorizadorFragmentos, ClasificadorFragmentos, modelo_neutral
Celda 50:
  one_hot_conector, regla_estructural, rasgos_fragmentos, matriz_rasgos, ComponenteEstLR, modelo_est_lr


## 12. Regularización dentro del ensamble v3b

Comparamos el ensamble original con una variante que reemplaza
únicamente su componente estructural por el regularizado.

Conservamos los demás componentes, sus parámetros y los pesos del
ensamble. Evaluamos ambas versiones sobre las mismas particiones.

Primero reconstruimos las definiciones necesarias del notebook v3b.
No repetimos la búsqueda de hiperparámetros ni la selección de pesos.

In [50]:
from scipy.sparse import hstack

from sklearn.base import ClassifierMixin, clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import StackingClassifier, VotingClassifier
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.preprocessing import StandardScaler

CLASES = np.array(LABELS)

# Conectores utilizados por los bloques de texto del modelo v3b.
FUERTES = [
    "pero al final", "pero", "aun asi", "aun con eso",
    "con todo", "al final de cuentas", "al final",
    "a fin de cuentas", "sin embargo", "no obstante",
    "de todos modos", "de todas formas", "lo que si",
    "ahora bien", "aunque",
]

DEBILES = ["eso si", "igual", "total", "en fin", "en cambio"]


def oraciones(t):
    return [
        s for s in re.split(r"(?<=[.!?])\s+", t) if s
    ] or [""]


def _regex_conectores(lista):
    alternativas = "|".join(
        re.escape(c)
        for c in sorted(lista, key=len, reverse=True)
    )
    return re.compile(
        r"(?:^|(?<=[.!?,;])\s*)(?:"
        + alternativas
        + r")\b"
    )


RE_FUERTE = _regex_conectores(FUERTES)
RE_DEBIL = _regex_conectores(DEBILES)
RE_ADVERSATIVO = _regex_conectores(FUERTES + DEBILES)


def desde_ultimo(texto, regex):
    coincidencias = list(regex.finditer(texto))
    return (
        texto[coincidencias[-1].start():]
        if coincidencias else ""
    )


class Bloques(TransformerMixin, BaseEstimator):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        textos = [normalizar(t) for t in X]
        frases = [oraciones(t) for t in textos]

        return pd.DataFrame({
            "completo": textos,
            "post": [
                desde_ultimo(t, RE_ADVERSATIVO) for t in textos
            ],
            "post_fuerte": [
                desde_ultimo(t, RE_FUERTE) for t in textos
            ],
            "post_debil": [
                desde_ultimo(t, RE_DEBIL) for t in textos
            ],
            "ultima": [o[-1] for o in frases],
            "penultima": [
                o[-2] if len(o) > 1 else "" for o in frases
            ],
        })


def tfidf_palabras():
    return TfidfVectorizer(
        ngram_range=(1, 2),
        sublinear_tf=True,
        min_df=2,
    )


def tfidf_caracteres():
    return TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(2, 5),
        sublinear_tf=True,
        min_df=3,
    )


def vectorizar(bloques, vectorizadores):
    return ColumnTransformer([
        (f"{bloque}_{i}", fabrica(), bloque)
        for bloque in bloques
        for i, fabrica in enumerate(vectorizadores)
    ])


BLOQUES_LINEALES = ["completo", "post", "ultima"]

BLOQUES_STACKING = [
    "completo", "post", "post_fuerte",
    "post_debil", "ultima", "penultima",
]


def modelo_lr_palabras():
    return Pipeline([
        ("bloques", Bloques()),
        (
            "vec",
            vectorizar(BLOQUES_LINEALES, [tfidf_palabras]),
        ),
        ("clf", LogisticRegression(C=5, max_iter=5000)),
    ])


def modelo_stacking():
    por_bloque = [
        (
            bloque,
            Pipeline([
                (
                    "vec",
                    vectorizar(
                        [bloque],
                        [tfidf_palabras, tfidf_caracteres],
                    ),
                ),
                ("clf", LogisticRegression(C=5, max_iter=5000)),
            ]),
        )
        for bloque in BLOQUES_STACKING
    ]

    meta = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=300,
        max_leaf_nodes=15,
        min_samples_leaf=40,
        early_stopping=False,
        random_state=0,
    )

    return Pipeline([
        ("bloques", Bloques()),
        (
            "stack",
            StackingClassifier(
                estimators=por_bloque,
                final_estimator=meta,
                stack_method="predict_proba",
                cv=StratifiedKFold(
                    5, shuffle=True, random_state=1
                ),
            ),
        ),
    ])


# Verificar la transformación y la construcción de los modelos.
ejemplo_bloques = (
    "La batería es excelente. "
    "Sin embargo, la cámara es decepcionante."
)

display(Bloques().transform([ejemplo_bloques]).T)

for fabrica in (modelo_lr_palabras, modelo_stacking):
    clone(fabrica())

print("Bloques, regresión logística y stacking preparados.")

,0
completo,"la bateria es excelente. sin embargo, la camar..."
post,"sin embargo, la camara es decepcionante."
post_fuerte,"sin embargo, la camara es decepcionante."
post_debil,
ultima,"sin embargo, la camara es decepcionante."
penultima,la bateria es excelente.


Bloques, regresión logística y stacking preparados.


### 12.1 Componente de clasificación por fragmentos

Reutilizamos la segmentación, el clasificador de fragmentos y el
componente est_lr del notebook modelo_v3b_estructural_polaridad.ipynb.

Este componente entrena su clasificador final con características
obtenidas mediante validación interna. Conservamos ese procedimiento
para evitar entrenarlo con predicciones de fragmentos obtenidas sobre
las mismas reseñas usadas para ajustar el clasificador de fragmentos.

Las siguientes celdas cargan únicamente las definiciones necesarias.
Este notebook depende del notebook original para reproducir el experimento.

In [51]:
import ast
import json

# Volver a leer el notebook original, sin modificarlo.
with SOURCE_NOTEBOOK.open(encoding="utf-8") as archivo:
    notebook_original = json.load(archivo)


def cargar_definiciones_fragmentos(numero_celda):
    celda = notebook_original["cells"][numero_celda - 1]
    assert celda["cell_type"] == "code"

    codigo = "".join(celda["source"])

    if numero_celda == 48:
        # El ejemplo final depende de variables del notebook original.
        marcador = "# Ejemplo de segmentación"
        assert marcador in codigo, (
            "La celda original cambió. Revisar antes de continuar."
        )
        codigo = codigo.split(marcador, maxsplit=1)[0]

    estructura = ast.parse(codigo)

    # Conservar funciones, clases y constantes.
    # En la celda 48 también se necesitan los bucles que construyen
    # el diccionario FR_TIPO de conectores.
    nodos = []

    for nodo in estructura.body:
        if isinstance(
            nodo,
            (ast.FunctionDef, ast.ClassDef, ast.Assign),
        ):
            nodos.append(nodo)

        elif numero_celda == 48 and isinstance(nodo, ast.For):
            assert (
                isinstance(nodo.target, ast.Name)
                and nodo.target.id == "_c"
            ), "Se encontró un bucle distinto al esperado."
            nodos.append(nodo)

        elif isinstance(nodo, ast.Expr):
            # Omitir las impresiones y demostraciones.
            continue

        else:
            raise ValueError(
                f"Revisar instrucción de la celda {numero_celda}: "
                f"{type(nodo).__name__}"
            )

    modulo = ast.Module(body=nodos, type_ignores=[])
    ast.fix_missing_locations(modulo)

    exec(
        compile(
            modulo,
            filename=f"{SOURCE_NOTEBOOK.name}:celda_{numero_celda}",
            mode="exec",
        ),
        globals(),
    )

    print(f"Definiciones de la celda {numero_celda}: cargadas.")


for numero in (48, 49, 50):
    cargar_definiciones_fragmentos(numero)

# Construir y clonar no entrena el modelo.
componente_demo = clone(modelo_est_lr())

print("\nParámetros de est_lr:")
print(componente_demo.get_params())

print("\nEjemplo de segmentación:")
for fragmento in segmentar(normalizar(
    "La batería es excelente, pero la cámara es decepcionante."
)):
    print(
        f"[{fragmento['tipo_con']}] "
        f"{fragmento['texto']}"
    )

Definiciones de la celda 48: cargadas.
Definiciones de la celda 49: cargadas.
Definiciones de la celda 50: cargadas.

Parámetros de est_lr:
{'C': 1.0, 'n_folds_internos': 5, 'semilla_interna': 1000}

Ejemplo de segmentación:
[ninguno] la bateria es excelente
[fuerte] pero la camara es decepcionante.


### 12.2 Comparación de ensambles con pesos fijos

Utilizamos los pesos registrados en el notebook v3b:

- Stacking: 0.20.
- Regresión logística por bloques: 0.30.
- Componente estructural: 0.30.
- Clasificador por fragmentos est_lr: 0.20.

La única diferencia entre los ensambles será la regularización del
componente estructural. Los demás componentes se entrenan una sola vez
por fold y se comparten entre ambas variantes.

Evaluamos cinco folds con dos semillas. Todas las predicciones de
validación se generan con modelos que no han entrenado con esas filas.

Los pesos y las reglas proceden de experimentos previos sobre estos
datos. Esta comparación es exploratoria, no una evaluación independiente
de todo el proceso de selección.

In [52]:
from datetime import datetime
from sklearn.preprocessing import LabelEncoder
import time

PESOS_COMPARACION = {
    "stacking": 0.20,
    "lr_palabras": 0.30,
    "estructural": 0.30,
    "est_lr": 0.20,
}

# Usar etiquetas numéricas como en la validación original de v3b.
codificador = LabelEncoder().fit(LABELS)
y_codificado = codificador.transform(y)
clases_ensamble = codificador.classes_

marca = datetime.now().strftime("%Y%m%d_%H%M%S")
carpeta_oof = ROOT / "results" / f"comparacion_ensamble_{marca}"
carpeta_oof.mkdir(parents=True, exist_ok=False)

COMPONENTES = [
    "lr_palabras",
    "estructural_original",
    "estructural_regularizado",
    "est_lr",
    "stacking",
]

oof_comparacion = {}
resultados_ensamble = []

inicio_comparacion = time.perf_counter()

for semilla, folds in PARTICIONES.items():
    probabilidades = {
        nombre: np.full((len(y), len(clases_ensamble)), np.nan)
        for nombre in COMPONENTES
    }
    cobertura = np.zeros(len(y), dtype=int)

    print(f"\n=== Semilla {semilla} ===", flush=True)

    for k, (tr_idx, va_idx) in enumerate(folds):
        # k comienza en 0, igual que en el notebook original.
        fabricas = {
            "lr_palabras": modelo_lr_palabras,
            "estructural_original": lambda: Pipeline([
                ("rasgos", RasgosPlantilla(
                    modo_polaridad="aprendida"
                )),
                ("clf", crear_clasificador("original")),
            ]),
            "estructural_regularizado": lambda: Pipeline([
                ("rasgos", RasgosPlantilla(
                    modo_polaridad="aprendida"
                )),
                ("clf", crear_clasificador("regularizado")),
            ]),
            "est_lr": lambda: modelo_est_lr(
                semilla_interna=1000 + k
            ),
            "stacking": modelo_stacking,
        }

        for nombre, fabrica in fabricas.items():
            print(
                f"Fold {k + 1}/5 — entrenando {nombre}...",
                flush=True,
            )
            inicio_modelo = time.perf_counter()

            candidato = fabrica()
            candidato.fit(X[tr_idx], y_codificado[tr_idx])

            orden = [
                list(candidato.classes_).index(i)
                for i in range(len(clases_ensamble))
            ]
            proba = candidato.predict_proba(X[va_idx])[:, orden]

            assert np.isfinite(proba).all()
            assert np.allclose(proba.sum(axis=1), 1)

            probabilidades[nombre][va_idx] = proba

            # Conservar cada resultado terminado.
            np.savez_compressed(
                carpeta_oof / (
                    f"semilla_{semilla}_fold_{k + 1}_{nombre}.npz"
                ),
                indices=va_idx,
                ids=train_df["id"].to_numpy()[va_idx],
                clases=clases_ensamble.astype(str),
                probabilidades=proba,
            )

            segundos = time.perf_counter() - inicio_modelo
            print(f"  Terminado en {segundos:.1f} s.", flush=True)

            del candidato

        cobertura[va_idx] += 1

        for variante in ("original", "regularizado"):
            mezcla = (
                PESOS_COMPARACION["stacking"]
                * probabilidades["stacking"][va_idx]
                + PESOS_COMPARACION["lr_palabras"]
                * probabilidades["lr_palabras"][va_idx]
                + PESOS_COMPARACION["est_lr"]
                * probabilidades["est_lr"][va_idx]
                + PESOS_COMPARACION["estructural"]
                * probabilidades[f"estructural_{variante}"][va_idx]
            )

            pred = clases_ensamble[mezcla.argmax(axis=1)]
            acc = accuracy_score(y[va_idx], pred)

            resultados_ensamble.append({
                "semilla": semilla,
                "fold": k + 1,
                "ensamble": variante,
                "accuracy": acc,
            })

            print(
                f"  Ensamble {variante}: {acc:.4f}",
                flush=True,
            )

        pd.DataFrame(resultados_ensamble).to_csv(
            carpeta_oof / "resultados_por_fold.csv",
            index=False,
        )

    assert (cobertura == 1).all()
    assert all(
        np.isfinite(p).all()
        for p in probabilidades.values()
    )
    oof_comparacion[semilla] = probabilidades

resultados_ensamble = pd.DataFrame(resultados_ensamble)

print(
    "\nTiempo total:",
    round((time.perf_counter() - inicio_comparacion) / 60, 1),
    "minutos",
)
print("Resultados guardados en:", carpeta_oof)


=== Semilla 42 ===
Fold 1/5 — entrenando lr_palabras...
  Terminado en 1.9 s.
Fold 1/5 — entrenando estructural_original...
  Terminado en 6.6 s.
Fold 1/5 — entrenando estructural_regularizado...
  Terminado en 5.9 s.
Fold 1/5 — entrenando est_lr...
  Terminado en 67.9 s.
Fold 1/5 — entrenando stacking...
  Terminado en 64.5 s.
  Ensamble original: 0.9125
  Ensamble regularizado: 0.9146
Fold 2/5 — entrenando lr_palabras...
  Terminado en 2.2 s.
Fold 2/5 — entrenando estructural_original...
  Terminado en 6.9 s.
Fold 2/5 — entrenando estructural_regularizado...
  Terminado en 6.4 s.
Fold 2/5 — entrenando est_lr...
  Terminado en 72.0 s.
Fold 2/5 — entrenando stacking...
  Terminado en 68.7 s.
  Ensamble original: 0.9108
  Ensamble regularizado: 0.9108
Fold 3/5 — entrenando lr_palabras...
  Terminado en 1.8 s.
Fold 3/5 — entrenando estructural_original...
  Terminado en 5.8 s.
Fold 3/5 — entrenando estructural_regularizado...
  Terminado en 6.1 s.
Fold 3/5 — entrenando est_lr...
  Termi

In [53]:
resumen_ensamble = (
    resultados_ensamble
    .groupby(["semilla", "ensamble"])["accuracy"]
    .mean()
    .unstack("ensamble")
)

resumen_ensamble["mejora_pp"] = 100 * (
    resumen_ensamble["regularizado"]
    - resumen_ensamble["original"]
)

display(resumen_ensamble.round(6))

ensamble,original,regularizado,mejora_pp
semilla,,,
42,0.916500,0.916250,-0.025000
2024,0.916583,0.915417,-0.116667


In [54]:
def modelo_estructural_regularizado():
    return Pipeline([
        (
            "rasgos",
            RasgosPlantilla(modo_polaridad="aprendida"),
        ),
        ("clf", crear_clasificador("regularizado")),
    ])


modelo_ensamble_regularizado = VotingClassifier(
    estimators=[
        ("stacking", modelo_stacking()),
        ("lr_palabras", modelo_lr_palabras()),
        ("estructural", modelo_estructural_regularizado()),
        ("est_lr", modelo_est_lr(semilla_interna=1000)),
    ],
    voting="soft",
    weights=[
        PESOS_COMPARACION["stacking"],
        PESOS_COMPARACION["lr_palabras"],
        PESOS_COMPARACION["estructural"],
        PESOS_COMPARACION["est_lr"],
    ],
    n_jobs=1,
)

inicio = time.perf_counter()
print("Entrenando el ensamble final...", flush=True)

modelo_ensamble_regularizado.fit(X, y)

print(
    "Entrenamiento terminado en",
    round((time.perf_counter() - inicio) / 60, 1),
    "minutos",
)

Entrenando el ensamble final...
Entrenamiento terminado en 2.8 minutos


In [55]:
import joblib
from datetime import datetime

eval_envio = pd.read_csv(DATA_DIR / "eval.csv")
plantilla_envio = pd.read_csv(DATA_DIR / "sample_submission.csv")

assert list(plantilla_envio.columns) == ["id", "answer"]
assert eval_envio[["id", "text"]].notna().all().all()
assert eval_envio["text"].str.strip().ne("").all()
assert plantilla_envio["id"].notna().all()
assert eval_envio["id"].is_unique
assert plantilla_envio["id"].is_unique
assert set(eval_envio["id"]) == set(plantilla_envio["id"])

pred_por_id = pd.Series(
    modelo_ensamble_regularizado.predict(
        eval_envio["text"].to_numpy()
    ),
    index=eval_envio["id"],
)

submission_ensamble = plantilla_envio.copy()
submission_ensamble["answer"] = (
    submission_ensamble["id"].map(pred_por_id)
)

assert submission_ensamble["answer"].notna().all()
assert set(submission_ensamble["answer"]).issubset(LABELS)

marca_envio = datetime.now().strftime("%Y%m%d_%H%M%S")
nombre_envio = f"ensamble_estructural_regularizado_{marca_envio}"

ruta_csv_ensamble = (
    ROOT / "submissions" / f"submission_{nombre_envio}.csv"
)
ruta_modelo_ensamble = (
    ROOT / "models" / f"modelo_{nombre_envio}.joblib"
)

for ruta in (ruta_csv_ensamble, ruta_modelo_ensamble):
    ruta.parent.mkdir(parents=True, exist_ok=True)
    assert not ruta.exists(), f"El archivo ya existe: {ruta}"

joblib.dump(
    modelo_ensamble_regularizado,
    ruta_modelo_ensamble,
    compress=3,
)
submission_ensamble.to_csv(ruta_csv_ensamble, index=False)

# Verificar los archivos guardados.
modelo_verificacion = joblib.load(ruta_modelo_ensamble)
csv_verificacion = pd.read_csv(ruta_csv_ensamble)

pred_verificacion = pd.Series(
    modelo_verificacion.predict(eval_envio["text"].to_numpy()),
    index=eval_envio["id"],
)

assert list(csv_verificacion.columns) == ["id", "answer"]
assert np.array_equal(
    csv_verificacion["id"].to_numpy(),
    plantilla_envio["id"].to_numpy(),
)
assert np.array_equal(
    csv_verificacion["answer"].to_numpy(),
    csv_verificacion["id"].map(pred_verificacion).to_numpy(),
)

print("OK: el modelo recargado reproduce el CSV.")
print("Filas:", len(csv_verificacion))
print("\nARCHIVO PARA SUBIR A KAGGLE:")
print(ruta_csv_ensamble)

OK: el modelo recargado reproduce el CSV.
Filas: 3000

ARCHIVO PARA SUBIR A KAGGLE:
/Users/alejandrobenavidesrubio/MLP1/submissions/submission_ensamble_estructural_regularizado_20261005_221405.csv


## 13. Resultado del ensamble regularizado en Kaggle

El ensamble con el componente estructural regularizado obtuvo un
Public Score de 0.90555, frente a 0.87111 del componente aislado.

No superó el mejor resultado previamente reportado por el equipo
(aproximadamente 0.9077).

En validación cruzada también quedó ligeramente por debajo del
ensamble original con ambas semillas. Por tanto, no se adopta esta
modificación como reemplazo del mejor modelo.

El experimento muestra que mejorar un componente por separado no
garantiza mejorar su combinación con otros modelos.

Se conservan el notebook, los resultados por fold, las probabilidades
OOF, el modelo y el submission para documentar la comparación.

## 14. Experimento: ensamble sin componente estructural

### Objetivo

Evaluar si retirar el componente estructural mejora el accuracy del
ensamble, manteniendo los otros modelos y sus proporciones relativas.

### Configuración

Redistribuimos los pesos restantes para que sumen 1:

| Componente | Peso original | Peso sin estructural |
|---|---:|---:|
| Stacking | 0.20 | 2/7 |
| Regresión logística por bloques | 0.30 | 3/7 |
| Clasificador por fragmentos est_lr | 0.20 | 2/7 |
| Estructural | 0.30 | 0 |

### Evaluación

Reutilizamos las probabilidades fuera de muestra (OOF) calculadas
con cinco folds y las semillas 42 y 2024. No se requiere entrenar
nuevamente los modelos.

Comparamos esta variante con el ensamble estructural original.
Una diferencia positiva en puntos porcentuales indica una mejora.

La evaluación es exploratoria porque reutiliza datos de validación
consultados en experimentos anteriores. Una mejora local no garantiza
una mejora en Kaggle.

In [56]:
comparacion_sin_estructural = []

for semilla, probabilidades in oof_comparacion.items():
    base_comun = (
        0.20 * probabilidades["stacking"]
        + 0.30 * probabilidades["lr_palabras"]
        + 0.20 * probabilidades["est_lr"]
    )

    proba_original = (
        base_comun
        + 0.30 * probabilidades["estructural_original"]
    )
    proba_sin_estructural = base_comun / 0.70

    pred_original = clases_ensamble[
        proba_original.argmax(axis=1)
    ]
    pred_sin_estructural = clases_ensamble[
        proba_sin_estructural.argmax(axis=1)
    ]

    acc_original = accuracy_score(y, pred_original)
    acc_sin_estructural = accuracy_score(y, pred_sin_estructural)

    comparacion_sin_estructural.append({
        "semilla": semilla,
        "original": acc_original,
        "sin_estructural": acc_sin_estructural,
        "diferencia_pp": 100 * (
            acc_sin_estructural - acc_original
        ),
    })

display(
    pd.DataFrame(comparacion_sin_estructural)
    .set_index("semilla")
    .round(6)
)

,original,sin_estructural,diferencia_pp
semilla,,,
42,0.916500,0.911917,-0.458333
2024,0.916583,0.911583,-0.500000


### Resultado y conclusión

Retirar el componente estructural redujo el accuracy en ambas semillas:

- Semilla 42: 0.916500 → 0.911917.
- Semilla 2024: 0.916583 → 0.911583.

Con estas configuraciones y pesos, el componente estructural aporta
información útil al ensamble. Se conserva y no se genera un submission
de la variante sin él.